In [1]:
import sys
sys.path.insert(0,'/mnt/AEA8F340A8F3059D/sportsbet/ai-engine')

In [2]:
from fastapi import APIRouter,HTTPException,Request
from fastapi.responses import StreamingResponse
from agents.nodes.chatassistant import chatAssistantStream
from agents.state import AgentState
from config.constants import INTENT_CHAT
from rag.retriever import retrieveChunks
from tools.mongodbtools import fetchUserProfile,fetchUserWallet
from models.schemas import validateChatRequest
import json,logging

In [3]:
router=APIRouter()
logger=logging.getLogger(__name__)

In [ ]:
@router.post("/chat")
async def chatStream(request:Request):
    body=await request.json()
    try:
        params=validateChatRequest(body)
    except Exception as e:
        raise HTTPException(status_code=400,detail=str(e))
    async def eventStream():
        try:
            yield f"data:{
                json.dumps({
                    'type':'start',
                    'sessionId':params['sessionId']
                })
            }"
            userProfile={}
            userWallet={}
            ragChunks=[]
            if params["userId"]:
                try:
                    userProfile=fetchUserProfile(params["userId"]) or {}
                    userWallet=fetchUserWallet(params["userId"]) or{}
                except Exception:
                    pass
            try:
                ragChunks=await retrieveChunks(params['query'],topK=3)
            except Exception:
                pass
            state=AgentState(
                query=params['query'],
                intent=INTENT_CHAT,
                confidence=1.0,
                context={
                    **params["context"],
                    "sessionId":params["sessionId"],
                },
                user_id=params["userId"],
                user_profile=userProfile,
                user_wallet=userWallet,
                rag_chunks=ragChunks
            )
            async for t in chatAssistantStream(state):
                yield f"data:{
                    json.dumps({
                        "type":'token',
                        'content':t
                    })
                }"
            yield f"data:{
                json.dumps({
                    'type':'done'
                })
            }"
        except Exception as e:
            logger.error(f"Chat stream error {e}")
            yield f"data:{
                json.dumps({
                    'type':'error',
                    'message':str(e)
                })
            }"
        return StreamingResponse(eventStream(),media_type="text/event-stream",headers={"Cache-Control":"no-cache","Connection":"keep-alive"})
        

SyntaxError: unterminated string literal (detected at line 10) (3716561060.py, line 10)